# CEBRA: Neural Manifolds and Position Decoding

Learn a representation of population activity that is useful for behavior decoding. We compare unsupervised PCA and UMAP with optional CEBRA, then decode a one-dimensional position from each embedding.

## Quick start

Run this notebook from top to bottom. By default it works offline with a small simulated place-cell dataset and produces PCA/UMAP plots plus decoding results. CEBRA and its demo data are an optional live extension later; leaving their switches off always uses the clearly labeled PCA fallback.

## Prerequisites

- Python 3.9+ and familiarity with NumPy arrays shaped `(time, neurons)`.
- Basic concepts of dimensionality reduction, train/test splits, and regression.
- A GPU is optional; the lightweight model below also runs on CPU.

## Setup

Install the minimal packages for the default offline analysis. The optional CEBRA section has its own install cell and is not needed for PCA, UMAP, or the simulated-data tutorial.

In [ ]:
# Default offline tutorial dependencies
!pip install -q umap-learn scikit-learn matplotlib

## Load neural activity and position

The default is a deterministic simulated place-cell population, so no network access or CEBRA installation is required. If you explicitly enable `USE_CEBRA_DEMO_DATA` after installing CEBRA, the loader attempts the package demo and returns to the simulation if it is unavailable. The simulation is an illustration, not a biological substitute for recorded data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error

rng = np.random.default_rng(7)

def simulated_place_cells(n_time=1800, n_neurons=48, seed=7):
    rng = np.random.default_rng(seed)
    position = np.mod(np.linspace(0, 4, n_time) + 0.08 * rng.standard_normal(n_time), 1)
    centers = np.linspace(0, 1, n_neurons, endpoint=False)
    circular_distance = np.abs(position[:, None] - centers[None, :])
    circular_distance = np.minimum(circular_distance, 1 - circular_distance)
    rates = 0.15 + 3.0 * np.exp(-(circular_distance / 0.11) ** 2)
    shared = 0.25 * rng.standard_normal((n_time, 3)) @ rng.standard_normal((3, n_neurons))
    activity = rng.poisson(np.clip(rates + shared, 0.05, None)).astype(float)
    return activity, position

USE_CEBRA_DEMO_DATA = False  # Optional: requires the optional CEBRA install below and may use the network.
activity, position = simulated_place_cells()
data_source = 'OFFLINE FALLBACK: simulated place-cell population'
if USE_CEBRA_DEMO_DATA:
    try:
        import cebra
        demo = cebra.datasets.init('rat-hippocampus-achilles')
        neural = next((getattr(demo, name) for name in ('neural', 'neural_data', 'data') if hasattr(demo, name)), None)
        labels = next((getattr(demo, name) for name in ('position', 'pos', 'labels') if hasattr(demo, name)), None)
        neural, labels = np.asarray(neural), np.asarray(labels).squeeze()
        if neural.ndim != 2 or labels.ndim != 1 or len(neural) != len(labels):
            raise ValueError('Demo data lacks matched 2-D neural data and 1-D position.')
        activity, position = neural[:4000], labels[:4000]
        data_source = 'CEBRA rat-hippocampus-achilles demo data'
    except Exception as error:
        print(f'Demo unavailable; retaining offline fallback: {error}')
activity = np.nan_to_num(activity, nan=0.0)
position = (position - position.min()) / (position.max() - position.min() + np.finfo(float).eps)
print(data_source, '| activity:', activity.shape, '| position:', position.shape)

## PCA and UMAP baselines

PCA preserves directions of maximal variance, which need not be behavioral variables. UMAP preserves local neighborhoods probabilistically. Color embeddings by held-out behavioral position only as a visual diagnostic; the quantitative comparison below uses the same held-out time samples for every method.

In [ ]:
from umap import UMAP

pca_embedding = PCA(n_components=3, random_state=7).fit_transform(activity)
umap_embedding = UMAP(n_components=3, n_neighbors=30, min_dist=0.15, random_state=7).fit_transform(activity)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, embedding, title in zip(axes, (pca_embedding, umap_embedding), ('PCA', 'UMAP')):
    image = ax.scatter(embedding[:, 0], embedding[:, 1], c=position, s=5, cmap='viridis')
    ax.set(title=title, xlabel='component 1', ylabel='component 2')
fig.colorbar(image, ax=axes, label='normalized position')
plt.show()

## Optional: train a behavior-conditioned CEBRA model

**Default behavior: PCA fallback.** CEBRA is not required for this tutorial. To enable it, run the optional install cell and set `RUN_CEBRA = True` below. CEBRA uses contrastive learning: nearby behavioral labels form positive pairs and other samples form negatives. If installation or fitting fails, the result remains a three-component PCA embedding and is prominently labeled as a fallback.

In [ ]:
# Optional CEBRA dependency; run only when enabling RUN_CEBRA.
# !pip install -q cebra


In [ ]:
RUN_CEBRA = False  # Default: use the PCA fallback shown below.
cebra_embedding = PCA(n_components=3, random_state=7).fit_transform(activity)
embedding_method = 'FALLBACK: PCA surrogate (CEBRA optional path disabled)'
if RUN_CEBRA:
    try:
        import cebra
        model = cebra.CEBRA(
            model_architecture='offset10-model', output_dimension=3,
            batch_size=min(256, len(activity)), max_iterations=300,
            conditional='time_delta', time_offsets=10,
            device='cuda_if_available', verbose=False,
        )
        model.fit(activity, position)
        cebra_embedding = model.transform(activity)
        embedding_method = 'CEBRA behavior-conditioned embedding'
    except Exception as error:
        print(f'CEBRA fitting failed; retaining FALLBACK PCA: {error}')


fig, ax = plt.subplots(figsize=(5, 4))
image = ax.scatter(cebra_embedding[:, 0], cebra_embedding[:, 1], c=position, s=5, cmap='viridis')
ax.set(title=embedding_method, xlabel='latent 1', ylabel='latent 2')
fig.colorbar(image, ax=ax, label='normalized position')
plt.show()

## Decode position on a chronological hold-out set

Randomly shuffling temporally autocorrelated data can overestimate performance. This simple tutorial uses an early training block and later test block, then fits the same k-nearest-neighbor regressor to each embedding. For real experiments, use blocked cross-validation with a gap matched to neural/behavioral autocorrelation.

In [ ]:
split = int(0.7 * len(position))
results = {}
for name, embedding in {'PCA': pca_embedding, 'UMAP': umap_embedding, 'CEBRA': cebra_embedding}.items():
    decoder = KNeighborsRegressor(n_neighbors=15, weights='distance')
    decoder.fit(embedding[:split], position[:split])
    predicted = decoder.predict(embedding[split:])
    results[name] = (r2_score(position[split:], predicted), mean_absolute_error(position[split:], predicted))

for name, (r2, mae) in results.items():
    print(f'{name:5s} | test R² = {r2: .3f} | mean absolute error = {mae:.3f}')

# Interpretation: compare estimates only within this dataset and split; do not claim
# statistical superiority without repeated, appropriately blocked validation.

## References

- Schneider, S., Lee, J. H., Mathis, M. W., & Mathis, A. (2023). Learnable latent embeddings for joint behavioral and neural analysis. *Nature*, 617, 360–368. https://doi.org/10.1038/s41586-023-06031-6
- CEBRA documentation and examples: https://cebra.ai/docs/
- McInnes, L., Healy, J., & Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction. https://arxiv.org/abs/1802.03426

## License

This notebook is licensed under the [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/). CEBRA and its example data retain their respective upstream licenses; cite the original work when using them.